# Paper Trading Runtime

This notebook turns the recorder into a paper runtime simulation. It does not call a private exchange API and it does not place live orders. It uses the selected quote-policy candidate, submits paper orders into an order-state machine, consumes public trades as if they were arriving live, applies queue-ahead, and writes the same hash-chained event log used by the recorder notebook.

The purpose is simulator validation: can a paper order manager reproduce the replay assumptions, and what changes when we enforce realistic quote replacement?


## Runtime Model

For each paper order we record:

$$
Decision \rightarrow Submit \rightarrow Ack \rightarrow \{Fill, CancelRequest, CancelAck\}
$$

A public trade can fill a paper bid when:

$$
side_{trade}=sell,\quad price_{trade}\leq price_{bid},\quad V_{through} > Q^{ahead}_{L2}
$$

A public trade can fill a paper ask when:

$$
side_{trade}=buy,\quad price_{trade}\geq price_{ask},\quad V_{through} > Q^{ahead}_{L2}
$$

The notebook runs two modes:

- `independent_orders`: matches the replay assumption where each quote decision is tested independently.
- `replace_by_side`: cancels the previous open paper order on the same side before submitting the next quote, which is closer to a real market maker.


In [ ]:
from __future__ import annotations

import json
import os
import shutil
import sys
from pathlib import Path

import IPython.display as ipd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from IPython.display import display


def find_project_root() -> Path:
    for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (candidate / "notebooks" / "quote_policy_optimizer.ipynb").exists():
            return candidate
    raise FileNotFoundError("Could not find project root containing notebooks/quote_policy_optimizer.ipynb")


PROJECT_ROOT = find_project_root()
NOTEBOOK_DIR = PROJECT_ROOT / "notebooks"
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.append(str(NOTEBOOK_DIR))

from paper_order_manager import PaperRuntimeConfig, run_historical_paper_runtime, runtime_summary  # noqa: E402
from paper_trade_recorder import (  # noqa: E402
    PaperTradeRecorder,
    build_order_plan,
    lifecycle_summary,
    replay_comparison,
    select_replay_orders,
    validate_event_log,
)

QUOTE_POLICY_NOTEBOOK = PROJECT_ROOT / "notebooks" / "quote_policy_optimizer.ipynb"
FEATURE_ROOT = Path(os.environ.get("MODL_WS_FEATURE_DIR", "/mnt/burner-archive/ws_features")).expanduser()
BAR_MINUTES = int(os.environ.get("MODL_BAR_MINUTES", "5"))

RUN_ID = os.environ.get("MODL_PAPER_RUNTIME_RUN_ID", pd.Timestamp.utcnow().strftime("runtime-%Y%m%dT%H%M%SZ"))
PAPER_OUTPUT_ROOT = Path(os.environ.get("MODL_PAPER_RUNTIME_OUTPUT_ROOT", "/tmp/modl_paper_trading_runtime")).expanduser()
ORDER_QTY = float(os.environ.get("MODL_STRATEGY_REPLAY_ORDER_QTY", "0.01"))
MAX_RUNTIME_ORDERS = int(os.environ.get("MODL_PAPER_RUNTIME_MAX_ORDERS", "40"))
ACK_LATENCY_MS = int(os.environ.get("MODL_PAPER_ACK_LATENCY_MS", "50"))
CANCEL_ACK_LATENCY_MS = int(os.environ.get("MODL_PAPER_CANCEL_ACK_LATENCY_MS", "100"))
RESET_RUNTIME_OUTPUT = os.environ.get("MODL_PAPER_RUNTIME_RESET_OUTPUT", "1") == "1"
SAVE_OUTPUTS = False

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 260)
pd.set_option("display.max_colwidth", 240)

PROJECT_ROOT, RUN_ID, PAPER_OUTPUT_ROOT, ORDER_QTY


## Load Policy And Market Data

We execute `quote_policy_optimizer.ipynb`, then pull the selected policy, detailed replay function, and public trade table from the nested strategy replay namespace.


In [ ]:
def quiet_display(*args, **kwargs):
    return None


def execute_notebook(path: Path) -> dict[str, object]:
    original_display = ipd.display
    original_cwd = Path.cwd()
    ipd.display = quiet_display
    namespace: dict[str, object] = {"__name__": "__paper_runtime_source__"}
    notebook = json.loads(path.read_text())
    try:
        os.chdir(PROJECT_ROOT)
        for cell in notebook["cells"]:
            if cell.get("cell_type") != "code":
                continue
            source = "".join(cell.get("source", []))
            if not source.strip():
                continue
            exec(compile(source, f"{path}:{cell.get('id', 'cell')}", "exec"), namespace)
            plt.close("all")
    finally:
        os.chdir(original_cwd)
        ipd.display = original_display
    return namespace


policy_ns = execute_notebook(QUOTE_POLICY_NOTEBOOK)
strategy_ns = policy_ns["strategy_ns"]
policy_frontier = policy_ns["policy_frontier"].copy()
paper_candidates = policy_ns["paper_candidates"].copy()
strategy_replay_by_ttl = policy_ns["strategy_replay_by_ttl"].copy()
frontier_cols = policy_ns["frontier_cols"]
public_trades = strategy_ns["trades"].copy()

display(policy_ns["gate_summary"])
display(paper_candidates[frontier_cols].head(10))


## Build Runtime Order Plan

The order plan comes from the best paper-gate row. If detailed replay rows are not already present for that exact latency and rebate scenario, we call `replay_policy` directly from the strategy replay namespace.


In [ ]:
if paper_candidates.empty:
    raise RuntimeError("No paper candidates available from quote_policy_optimizer.ipynb")

selected_candidate = paper_candidates.sort_values("implementation_score_bps", ascending=False).iloc[0].to_dict()
selected_policy = str(selected_candidate["policy"])
selected_latency_ms = int(selected_candidate["latency_ms"])
selected_ttl_ms = int(selected_candidate["ttl_ms"])
selected_rebate_bps = float(selected_candidate["maker_rebate_bps"])

selected_replay_orders = select_replay_orders(
    strategy_replay_by_ttl,
    selected_policy,
    selected_latency_ms,
    selected_ttl_ms,
    selected_rebate_bps,
    max_orders=MAX_RUNTIME_ORDERS,
)
detailed_replay_source = "strategy_replay_by_ttl"
if selected_replay_orders.empty:
    selected_replay_orders = strategy_ns["replay_policy"](
        selected_policy,
        strategy_ns["policy_inputs"][selected_policy],
        selected_latency_ms,
        selected_ttl_ms,
        selected_rebate_bps,
    ).sort_values(["decision_mts", "side", "quote_distance_bps"]).head(MAX_RUNTIME_ORDERS).reset_index(drop=True)
    detailed_replay_source = "strategy_replay.replay_policy"

order_plan = build_order_plan(selected_replay_orders, RUN_ID, ORDER_QTY)
if order_plan.empty:
    raise RuntimeError("Selected paper candidate produced no runtime order plan")

plan_summary = pd.DataFrame(
    [
        {
            "run_id": RUN_ID,
            "policy": selected_policy,
            "latency_ms": selected_latency_ms,
            "ttl_ms": selected_ttl_ms,
            "maker_rebate_bps": selected_rebate_bps,
            "orders": len(order_plan),
            "detailed_replay_source": detailed_replay_source,
            "expected_full_fills": int(order_plan["event_full_fill"].sum()),
            "expected_partial_or_full": int((order_plan["event_partial_qty"].fillna(0.0) > 0).sum()),
            "expected_l2_coverage_rate": float(order_plan["l2_price_covered"].mean()),
            "expected_ev_after_rebate_bps": float(order_plan["event_value_after_rebate_bps"].mean()),
        }
    ]
)

display(pd.DataFrame([selected_candidate])[frontier_cols])
display(plan_summary)
display(order_plan.head(20))


## Run Paper Runtime Modes

`independent_orders` should be closest to the replay. `replace_by_side` is closer to a production market maker because it avoids stacking overlapping quotes on the same side.


In [ ]:
if RESET_RUNTIME_OUTPUT and PAPER_OUTPUT_ROOT.exists():
    shutil.rmtree(PAPER_OUTPUT_ROOT)

runtime_modes = {
    "independent_orders": PaperRuntimeConfig(
        ack_latency_ms=ACK_LATENCY_MS,
        cancel_ack_latency_ms=CANCEL_ACK_LATENCY_MS,
        replace_existing_by_side=False,
        fill_during_cancel_pending=True,
    ),
    "replace_by_side": PaperRuntimeConfig(
        ack_latency_ms=ACK_LATENCY_MS,
        cancel_ack_latency_ms=CANCEL_ACK_LATENCY_MS,
        replace_existing_by_side=True,
        fill_during_cancel_pending=True,
    ),
}

runtime_results: dict[str, dict[str, object]] = {}
for mode_name, config in runtime_modes.items():
    mode_run_id = f"{RUN_ID}-{mode_name}"
    recorder = PaperTradeRecorder(
        PAPER_OUTPUT_ROOT,
        mode_run_id,
        venue="hibachi",
        symbol="BTC/USDT-P",
        strategy=selected_policy,
    )
    events = run_historical_paper_runtime(recorder, order_plan, public_trades, config)
    validation = validate_event_log(events)
    lifecycle = lifecycle_summary(events)
    comparison = replay_comparison(order_plan, lifecycle)
    summary = runtime_summary(events, lifecycle)
    summary.insert(0, "mode", mode_name)
    summary["event_log_path"] = str(recorder.path)
    if not validation["passed"].all():
        display(validation)
        raise RuntimeError(f"Paper runtime validation failed for {mode_name}")
    runtime_results[mode_name] = {
        "events": events,
        "validation": validation,
        "lifecycle": lifecycle,
        "comparison": comparison,
        "summary": summary,
        "event_log_path": recorder.path,
    }

runtime_summary_table = pd.concat([result["summary"] for result in runtime_results.values()], ignore_index=True)
comparison_summary = pd.DataFrame(
    [
        {
            "mode": mode_name,
            "orders": len(result["comparison"]),
            "expected_full_fills": int(result["comparison"]["event_full_fill"].sum()) if len(result["comparison"]) else 0,
            "recorded_full_fills": int(result["comparison"]["recorded_full_fill"].sum()) if len(result["comparison"]) else 0,
            "fill_match_rate": float(result["comparison"]["fill_match"].mean()) if len(result["comparison"]) else np.nan,
            "mean_qty_gap": float(result["comparison"]["qty_gap"].mean()) if len(result["comparison"]) else np.nan,
            "median_submit_to_ack_ms": float(result["lifecycle"]["submit_to_ack_ms"].median()) if len(result["lifecycle"]) else np.nan,
            "median_cancel_latency_ms": float(result["lifecycle"]["cancel_latency_ms"].median()) if len(result["lifecycle"]) else np.nan,
        }
        for mode_name, result in runtime_results.items()
    ]
)

display(runtime_summary_table)
display(comparison_summary)


## Lifecycle Detail

The lifecycle table lets us see whether the paper order manager is mostly cancelling, filling, or getting partial fills before cancel acknowledgement.


In [ ]:
lifecycle_tables = []
for mode_name, result in runtime_results.items():
    frame = result["lifecycle"].copy()
    frame.insert(0, "mode", mode_name)
    lifecycle_tables.append(frame)
runtime_lifecycle = pd.concat(lifecycle_tables, ignore_index=True)

display(runtime_lifecycle.groupby(["mode", "status"], as_index=False).size())
display(runtime_lifecycle.head(40))


## Runtime Diagnostics

These are the first paper-runtime charts to monitor: status counts, fills versus replay, ack/cancel latencies, and queue/EV relation.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

sns.barplot(data=runtime_summary_table, x="mode", y="events", ax=axes[0, 0])
axes[0, 0].set_title("Runtime Event Count")
axes[0, 0].tick_params(axis="x", rotation=15)

sns.countplot(data=runtime_lifecycle, x="mode", hue="status", ax=axes[0, 1])
axes[0, 1].set_title("Order Status By Runtime Mode")
axes[0, 1].tick_params(axis="x", rotation=15)

latency_frame = runtime_lifecycle[["mode", "submit_to_ack_ms", "cancel_latency_ms", "submit_to_fill_ms"]].melt(
    id_vars="mode",
    var_name="latency_type",
    value_name="latency_ms",
).dropna()
sns.boxplot(data=latency_frame, x="latency_type", y="latency_ms", hue="mode", ax=axes[1, 0])
axes[1, 0].set_title("Runtime Timing")
axes[1, 0].tick_params(axis="x", rotation=20)

comparison_plot = []
for mode_name, result in runtime_results.items():
    frame = result["comparison"].copy()
    frame.insert(0, "mode", mode_name)
    comparison_plot.append(frame)
comparison_plot = pd.concat(comparison_plot, ignore_index=True)
sns.scatterplot(
    data=comparison_plot,
    x="l2_queue_ahead_qty",
    y="event_value_after_rebate_bps",
    hue="recorded_full_fill",
    style="mode",
    ax=axes[1, 1],
)
axes[1, 1].axhline(0, color="black", linewidth=1)
axes[1, 1].set_title("Runtime Fills Versus Replay EV And Queue")

plt.tight_layout()


## PM Interpretation

If `independent_orders` agrees with replay but `replace_by_side` materially reduces fills, then the replay was testing quote ideas independently, not a deployable order manager. That is expected. The production strategy should be evaluated in replacement mode because a market maker normally keeps one quote per side and continuously cancels/replaces stale quotes.

The next integration step is to feed this runtime from live public market data and live model decisions instead of historical replay rows. The event schema should not change.


## Optional Save

Set `SAVE_OUTPUTS = True` to write runtime summaries and event logs under `MODL_WS_FEATURE_DIR/paper_trading_runtime/bar_<N>m/run_id=<RUN_ID>`.


In [ ]:
if SAVE_OUTPUTS:
    out_dir = FEATURE_ROOT / "paper_trading_runtime" / f"bar_{BAR_MINUTES}m" / f"run_id={RUN_ID}"
    out_dir.mkdir(parents=True, exist_ok=True)
    order_plan.to_parquet(out_dir / "order_plan.parquet")
    plan_summary.to_parquet(out_dir / "plan_summary.parquet")
    runtime_summary_table.to_parquet(out_dir / "runtime_summary.parquet")
    comparison_summary.to_parquet(out_dir / "comparison_summary.parquet")
    runtime_lifecycle.to_parquet(out_dir / "runtime_lifecycle.parquet")
    for mode_name, result in runtime_results.items():
        mode_dir = out_dir / mode_name
        mode_dir.mkdir(parents=True, exist_ok=True)
        result["events"].to_parquet(mode_dir / "paper_events.parquet")
        result["validation"].to_parquet(mode_dir / "validation.parquet")
        result["lifecycle"].to_parquet(mode_dir / "lifecycle.parquet")
        result["comparison"].to_parquet(mode_dir / "replay_comparison.parquet")
        shutil.copy2(result["event_log_path"], mode_dir / "paper_events.jsonl")
    print(f"wrote paper runtime outputs to {out_dir}")
else:
    print("SAVE_OUTPUTS is false; runtime JSONL remains under", PAPER_OUTPUT_ROOT)
